**candidate number: 26508**  
**Kernal: python 3.10.6**  
**purpose: collect information related to the topic/keyword "democracy" by scrapping on Wikipedia**

#### 🤖In the **part 2 code cell** in this jupyter notebook, I could only obtain the first half of the text for the **description** content.  
#### I reported my problem to Chatgpt, and it taught me that I should use **getall()** instead of **get())** to obtain the full-text data under the **div.searchresult card** 

**part 0**: Setups

In [ ]:
import requests
from scrapy import Selector
import pandas as pd
from pprint import pprint
from tqdm.notebook import tqdm

**part 1**: Define URLs and identify myself, test connections

from the Wikipedia I know the part changes in the URL is the **"offset = 'number'."**

In [ ]:
base_url_part1='https://en.wikipedia.org/w/index.php?advancedSearch-current={%22fields%22:{%22intitle%22:%22democracy%22}}&limit=100&offset='
base_url_part2='&profile=default&search=intitle%3Ademocracy&title=Special:Search&ns0=1'

In [ ]:
url = base_url_part1 + "100" + base_url_part2
headers = {'User-Agent': 'Web scraping practice  (c.feng6@lse.ac.uk)'}
response = requests.get(url, headers = headers)
if response.status_code == 200:
    print ('web connection successful, status code:', response)
else:
    print ('fail to connect to Wikipedia, could not proceed, status code:', response)
pprint(response.text[:1100])

In [ ]:
response = requests.get(base_url_part1 + "100" + base_url_part2)
sel = Selector(text = response.text)

Lets see how many headings there are in the page, as a duble-check to ensure have attained the precise URL

In [ ]:
headers_in_wikipedia = sel.css('div.mw-search-result-heading').getall()
print('There are', len(headers_in_wikipedia), 'results for headings per page.') 

the result is 100 per page. Expected, because I initially set the maximum number as 100.

**part 2**: showing paths of information

Each type of information will have its specific path. Find them, then define the route.

In [ ]:
def parse_democracy_wikipedia(democracy):
    
    democracy_wikipedia = {}
    democracy_wikipedia['title'] = democracy.css('div.mw-search-result-heading ::text').get()
    democracy_wikipedia['description'] = democracy.css('div.searchresult ::text').getall()
    democracy_wikipedia['file size and date'] = democracy.css('div.mw-search-result-data ::text').getall()
    democracy_wikipedia['additional keywords'] = democracy.css('span.searchalttitle ::text').getall()
    democracy_wikipedia['link'] = democracy.css('a ::attr(href)').get()
# ↑ Use get(), but not getall() here because I only want parent links. 
    democracy_wikipedia['derived_link'] = democracy.xpath("//span[@class='searchalttitle']/a/@href").get()
# ↑ Same logic, now I want children links only. Xpath is a better option here.
    return democracy_wikipedia

**part3**: writing loops, build dataframe

there are 1375 results for my keyword "democracy". Since each page has 100 results, there will be 14 pages.

Therefore, the variable index should be in series: 0, 100, 200, 300... 1400.

In [ ]:
all_democracy_data= []
for page_num in range(0, 14):
    url_2 = base_url_part1 + str(100 * page_num) + base_url_part2 # loop for scraping every url
    response = requests.get(url_2)
    if response.status_code == 200:
        print ('successfully connected to Wikipedia for page', page_num, 'status code:', response.status_code, '[Scraping now......]')
    else:
        print ('fail to connect to Wikipedia, could not proceed', page_num, 'status code:', response.status_code)
    sel = Selector(text = response.text)
    data_place = sel.css('td.searchResultImage-text')
    democracy_data = [parse_democracy_wikipedia(democracy) for democracy in data_place]
    all_democracy_data.extend(democracy_data)
    
    
    
    

check if it correctly works:

In [ ]:
len(all_democracy_data)

In [ ]:
pprint(all_democracy_data)

**part4**: save the output

In [ ]:
df = pd.DataFrame(all_democracy_data)
df.to_csv(r'D:\ds105a-2023-w07-summative-oche32\initial data\brief_wikipedia_data.csv', index=False)